# INIT

In [0]:
%sql
SHOW TABLES IN workspace.shop_flow_silver;

In [0]:
%sql
DESCRIBE TABLE workspace.shop_flow_silver.customers;
DESCRIBE TABLE workspace.shop_flow_silver.orders;
DESCRIBE TABLE workspace.shop_flow_silver.order_items;
DESCRIBE TABLE workspace.shop_flow_silver.order_payments;
DESCRIBE TABLE workspace.shop_flow_silver.order_reviews;
DESCRIBE TABLE workspace.shop_flow_silver.products;
DESCRIBE TABLE workspace.shop_flow_silver.sellers;

## GOLD_SCHEMA

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.shop_flow_gold;


# DIM_CUSTOMER

In [0]:
# create the dim_customer sql query

customer_query = """
    SELECT
        ROW_NUMBER() OVER(ORDER BY customer_id) AS customer_key,
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state
    FROM workspace.shop_flow_silver.customers
"""

In [0]:
# save the dim_customer table

dim_customer = spark.sql(customer_query)
dim_customer.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.dim_customer")
dim_customer = spark.table("workspace.shop_flow_gold.dim_customer")
dim_customer.display()

# DIM_PRODUCT

In [0]:
# create the dim_customer sql query

products_query = """
    SELECT
        ROW_NUMBER() OVER( ORDER BY product_id) AS product_key,
        *
    FROM workspace.shop_flow_silver.products
"""

In [0]:
# save the dim_product table

dim_product = spark.sql(products_query)
dim_product.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.dim_product")
dim_product = spark.table("workspace.shop_flow_gold.dim_product")
dim_product.display()

# DIM_SELLER

In [0]:

# create the dim_seller sql query

seller_query= """
    SELECT 
        ROW_NUMBER() OVER( ORDER BY seller_id) AS seller_key,
        *
    FROM workspace.shop_flow_silver.sellers
"""

In [0]:
# save the dim_product table

dim_seller = spark.sql(seller_query)
dim_seller.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.dim_seller")
dim_seller = spark.table("workspace.shop_flow_gold.dim_seller")
dim_seller.display()

# DIM_ORDER

In [0]:
%sql
SELECT
    MIN(TO_DATE(order_purchase_timestamp)) AS min_purchase_date,
    MAX(TO_DATE(order_purchase_timestamp)) AS max_purchase_date
FROM workspace.shop_flow_silver.orders;

In [0]:
date_query = """
    WITH date_range AS (
        SELECT EXPLODE(
            SEQUENCE(
                DATE '2016-09-04',
                DATE '2018-10-17',
                INTERVAL 1 DAY
            )
        ) AS full_date
    )

    SELECT
        CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
        full_date,
        DAY(full_date) AS day,
        DAYOFWEEK(full_date) AS day_of_week,
        DATE_FORMAT(full_date, 'EEEE') AS day_name,
        MONTH(full_date) AS month,
        DATE_FORMAT(full_date, 'MMMM') AS month_name,
        QUARTER(full_date) AS quarter,
        YEAR(full_date) AS year,
        CASE
            WHEN DAYOFWEEK(full_date) IN (1, 7) THEN TRUE
            ELSE FALSE
        END AS is_weekend
    FROM date_range;
"""



In [0]:
# save the dim_order table

dim_date = spark.sql(date_query)
dim_date.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.dim_date")
dim_date = spark.table("workspace.shop_flow_gold.dim_date")
dim_date.display()

# FACT_ORDER_ITEMS

In [0]:
order_items_query = """
    SELECT 
        oi.order_id,
        oi.order_item_id,
        p.product_key,
        s.seller_key,
        c.customer_key,
        CAST(DATE_FORMAT(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS date_key,
        o.order_purchase_timestamp,
        oi.shipping_limit_date,
        oi.price,
        oi.freight_value
    FROM workspace.shop_flow_silver.order_items oi
    LEFT JOIN workspace.shop_flow_silver.orders o
    ON oi.order_id = o.order_id
    LEFT JOIN workspace.shop_flow_gold.dim_product p
    ON oi.product_id = p.product_id
    LEFT JOIN workspace.shop_flow_gold.dim_seller s
    ON oi.seller_id = s.seller_id  
    LEFT JOIN workspace.shop_flow_gold.dim_customer c
    ON o.customer_id = c.customer_id
"""


In [0]:
# save the fact_order_items table

fact_order_items = spark.sql(order_items_query)
fact_order_items.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.fact_order_items")
fact_order_items = spark.table("workspace.shop_flow_gold.fact_order_items")
fact_order_items.display()


In [0]:
%sql
SELECT COUNT(*)
FROM workspace.shop_flow_gold.fact_order_items;

In [0]:
fact_order_items.groupBy("order_id","order_item_id").count().filter("count > 1").display()

# FACT_ORDERS

In [0]:
orders_query ="""
    SELECT 
        o.order_id,
        c.customer_key,
        CAST(DATE_FORMAT(o.order_purchase_timestamp, 'yyyyMMdd') AS INT) AS purchase_date_key,
        CAST(DATE_FORMAT(o.order_approved_at, 'yyyyMMdd') AS INT) AS approved_date_key,
        CAST(DATE_FORMAT(o.order_delivered_carrier_date, 'yyyyMMdd') AS INT) AS delivered_carrier_date_key,
        CAST(DATE_FORMAT(o.order_delivered_customer_date, 'yyyyMMdd') AS INT) AS delivered_customer_date_key,
        CAST(DATE_FORMAT(o.order_estimated_delivery_date, 'yyyyMMdd') AS INT) AS estimated_delivery_date_key,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date
    FROM workspace.shop_flow_silver.orders o
    LEFT JOIN workspace.shop_flow_gold.dim_customer c
    ON o.customer_id = c.customer_id
"""

In [0]:
# save the fact_orders table

fact_orders = spark.sql(orders_query)
fact_orders.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.fact_orders")
fact_orders = spark.table("workspace.shop_flow_gold.fact_orders")
fact_orders.display()

# FACT_ORDER_PAYMENTS

In [0]:
payment_query = """
    SELECT
        *
    FROM workspace.shop_flow_silver.order_payments
"""

In [0]:
# save the fact_order_payments table

fact_order_payments = spark.sql(payment_query)
fact_order_payments.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.fact_order_payments")
fact_order_payments = spark.table("workspace.shop_flow_gold.fact_order_payments")
fact_order_payments.display()

# FACT_ORDER_REVIEWS

In [0]:
review_query = """
    SELECT
        *
    FROM workspace.shop_flow_silver.order_reviews
"""

In [0]:
# save the fact_order_reviews table

fact_order_reviews = spark.sql(review_query)
fact_order_reviews.write.mode("overwrite").saveAsTable("workspace.shop_flow_gold.fact_order_reviews")
fact_order_reviews = spark.table("workspace.shop_flow_gold.fact_order_reviews")
fact_order_reviews.display()